In [8]:
#working code
import os
from dotenv import load_dotenv

# 1. CORE IMPORTS
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain_core.documents import Document

# New Agent Package (fixes the Deprecation Warning)
from langchain.agents import create_agent 

# Contextual Compression (to extract only answer-relevant sentences)
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

# 2. INITIALIZATION
load_dotenv(".env")

# Model 2.5 is more stable for tool-calling in the free tier
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=0)

# Local embeddings save you from "429 Quota Exhausted" errors
print("🚀 Loading local embedding model (CPU)...")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

DB_PATH = "./chroma_db_local"

# 3. DOCUMENT LOADING & PERSISTENCE
if os.path.exists(DB_PATH):
    print("✨ Found existing knowledge base. Loading...")
    vectorstore = Chroma(persist_directory=DB_PATH, embedding_function=embeddings)
else:
    print("📚 Creating new knowledge base from 'sample.txt'...")
    try:
        with open("sample.txt", "r", encoding="utf-8") as f:
            raw_text = f.read()
        
        doc = Document(page_content=raw_text, metadata={"source": "sample.txt"})
        vectorstore = Chroma.from_documents(
            documents=[doc],
            embedding=embeddings,
            persist_directory=DB_PATH
        )
        print("✅ Indexing complete.")
    except FileNotFoundError:
        print("❌ Error: 'sample.txt' not found.")
        vectorstore = Chroma(embedding_function=embeddings)

# 4. COMPRESSION RETRIEVER (Filters text for the LLM)
compressor = LLMChainExtractor.from_llm(llm)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=vectorstore.as_retriever(search_kwargs={"k": 2})
)

# 5. AGENT TOOL
@tool
def ask_docs(question: str) -> str:
    """Queries the local documentation about LangChain."""
    # We use .invoke() as per the 2026 standard
    compressed_docs = compression_retriever.invoke(question)
    
    if not compressed_docs:
        return "The document doesn't contain that information."
    
    return "\n".join([d.page_content for d in compressed_docs])

# 6. INITIALIZE AGENT
agent_executor = create_agent(llm, [ask_docs])

# 7. FIXED RUNNER (Handles node-based streaming)
def run_agent(query: str):
    print(f"\n--- User: {query} ---")
    inputs = {"messages": [HumanMessage(content=query)]}
    
    # We stream 'updates' to see each step of the thinking process
    for chunk in agent_executor.stream(inputs, stream_mode="updates"):
        # The key can be 'model' or 'agent' depending on your package version
        node_name = "agent" if "agent" in chunk else "model" if "model" in chunk else None
        
        if node_name:
            msg = chunk[node_name]["messages"][-1]
            if msg.tool_calls:
                print(f"🤔 Thinking... I need to search the document for: '{msg.tool_calls[0]['args']['question']}'")
            elif msg.content:
                print(f"\n🤖 FINAL ANSWER:\n{msg.content}")
        
        elif "tools" in chunk:
            print("👁️ Observation: Found relevant snippets in the knowledge base.")

# 8. TEST EXECUTION
run_agent("Who created LangChain and what is it used for?")

🚀 Loading local embedding model (CPU)...
✨ Found existing knowledge base. Loading...

--- User: Who created LangChain and what is it used for? ---
🤔 Thinking... I need to search the document for: 'Who created LangChain and what is it used for?'
👁️ Observation: Found relevant snippets in the knowledge base.

🤖 FINAL ANSWER:
"LangChain was created by Harrison Chase and it is a framework for building applications with LLMs."


In [12]:
# Change from 'langchain.chains' to 'langchain_classic.chains'
from langchain_classic.chains import create_history_aware_retriever, create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

In [14]:
#contextualcompression without agent
import os
from dotenv import load_dotenv

# Modern Model Imports
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.messages import HumanMessage, AIMessage

# Legacy/Classic Chain Imports for 2026
from langchain_classic.chains import create_history_aware_retriever, create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

load_dotenv(".env")

# 1. BRAIN & VECTOR DB
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=0)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory="./chroma_db_local", embedding_function=embeddings)

# 2. COMPRESSION SETUP
compressor = LLMChainExtractor.from_llm(llm)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=vectorstore.as_retriever(search_kwargs={"k": 2})
)

# 3. CONTEXTUALIZING THE QUESTION
# This re-writes the question if it refers to history (e.g., "Who created it?")
condense_q_system_prompt = "Given a chat history and the latest user question, formulate a standalone question. Do NOT answer the question."
condense_q_prompt = ChatPromptTemplate.from_messages([
    ("system", condense_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])
history_aware_retriever = create_history_aware_retriever(llm, compression_retriever, condense_q_prompt)

# 4. FINAL QA CHAIN
qa_system_prompt = "Answer the user's question using only the following context:\n\n{context}"
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", qa_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])
question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)
rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)

# 5. EXECUTION LOOP
chat_history = []

def chat_with_docs(user_input):
    global chat_history
    print(f"\n💬 User: {user_input}")
    
    # Run the chain
    result = rag_chain.invoke({"input": user_input, "chat_history": chat_history})
    
    # Print and update history
    print(f"🤖 Gemini: {result['answer']}")
    chat_history.extend([
        HumanMessage(content=user_input),
        AIMessage(content=result['answer']),
    ])

# Test
chat_with_docs("What is LangChain?")
chat_with_docs("Who created it?")


💬 User: What is LangChain?
🤖 Gemini: LangChain is a framework for building applications with LLMs.

💬 User: Who created it?
🤖 Gemini: LangChain was created by Harrison Chase.
